# 🧪 Model Experiments (Reference Only)

These models were explored during development to benchmark performance, assess generalization, and validate modeling choices. While some achieved competitive scores, they did not outperform XGBoost in terms of interpretability, robustness, and integration with geospatial analysis.

Full training pipelines and hyperparameter tuning are available in [`04a_model_experiments.ipynb`](./04a_model_experiments.ipynb).

---

### ➤ Model Comparison Summary (Best Cross-Validated R²)

| Model                      | R² Score | Feature Set     | Notes |
|----------------------------|----------|------------------|-------|
| **XGBoost Regressor**      | **0.536** | `corr_features`  | ✅ Final selected model (robust, interpretable, scalable) |
| KNeighbors Regressor (KNN) | 0.543     | `corr_features`  | Highest R², but sensitive to distance and scaling |
| Support Vector Regressor   | 0.503     | `corr_features`  | Performed well on selected features |
| Random Forest Regressor    | 0.495     | `corr_features`  | Strong non-linear baseline |
| ElasticNet Regression      | 0.433     | `10_features`    | Regularized linear model |
| Ridge Regression           | 0.436     | `10_features`    | Improved over baseline with L2 penalty |
| Linear Regression          | 0.171     | `10_features`    | Basic baseline model |
| MLP Regressor (Neural Net) | 0.265     | `10_features`    | Underfit, not optimal on tabular data |

---

➡️ While **KNN** and **SVR** achieved high R² scores, **XGBoost** was selected as the final model due to:
- Consistent performance across counties
- Better explainability (tree-based features)
- Seamless integration with the geospatial and underinsurance pipeline

---
### 📦 Load Packages 


In [3]:
! pip install -q xgboost

import geopandas as gpd
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, train_test_split, RepeatedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


---
### 📂 Data Loading


In [5]:
# Load from Parquet (preserves full Python types)
df = gpd.read_parquet("../output/gdf_ca_scored_v4.parquet")

print("Shape:", df.shape)
df.head()


Shape: (58, 72)


,county,total_population,median_income,age_0_17,age_18_64,age_65_up,age_0_17_pct,age_18_64_pct,age_65_up_pct,below_poverty,...,pre_1980_pct,built_1980_1999_pct,built_2000_up_pct,underinsured_fault_zone,underinsured_quake_zone,age_gap,seismic_risk_score,vulnerability_score,insurance_gap_score,underinsurance_risk_score
0,Sierra,2731,60000,349,1440,942,12.8,52.7,34.5,325,...,57.27,26.39,16.34,False,False,21.7,0.363,0.430,0.381,0.391
1,Sacramento,1584047,88724,365667,981913,236467,23.1,62.0,14.9,197472,...,49.21,29.36,21.42,False,False,-8.2,0.217,0.466,0.517,0.400
2,Santa Barbara,443975,95977,99608,272385,71982,22.4,61.4,16.2,58387,...,60.82,24.71,14.47,False,False,-6.2,0.527,0.540,0.407,0.491
3,Calaveras,45995,79877,7768,25007,13220,16.9,54.4,28.7,6083,...,33.46,42.18,24.36,False,False,11.8,0.180,0.247,0.421,0.283
4,Ventura,838259,107327,185795,511149,141315,22.2,61.0,16.9,74630,...,56.30,27.96,15.74,False,False,-5.3,0.517,0.398,0.325,0.414


---
### Target & Feature Selection


In [7]:
# Define target
TARGET = "eq_takeup_pct"

# Full Feature Set (based on domain & modeling intuition)
selected_features_20 = [
    'centroid_to_fault_km', 'fault_length_km', 'n_faults_intersecting',
    'eq_count', 'eq_max_mag', 'below_poverty_pct', 'pre_1980_pct',
    'renter_pct', 'owner_pct', 'age_gap', 'migrant_pct', 'seasonal_pct',
    'age_65_up_pct', 'age_0_17_pct', 'homeowners_coverage_pct',
    'modeled_loss_ratio_2018_pct', 'median_income', 'median_home_value',
    'fema_funding_earthquake', 'intersects_fault'
]

# Compact Feature Set (top 10 features from experimentation)
selected_features_10 = [
    "fault_length_km", "centroid_to_fault_km", "homeowners_coverage_pct",
    "eq_count", "median_home_value", "n_faults_intersecting",
    "age_0_17_pct", "seasonal_pct", "age_gap", "modeled_loss_ratio_2018_pct"
]

# Correlation-Based Feature Set
selected_features_corr = [
    'median_income', 'age_0_17_pct', 'age_18_64_pct', 'below_poverty_pct', 
    'american_indian_alone_pct', 'for_sale_pct', 'seasonal_pct', 'migrant_pct', 
    'other_vacant_pct', 'median_home_value', 'new_voluntary', 'renewed_voluntary', 
    'new_fair', 'homeowners_coverage_pct', 'direct_insured_pml_2018', 
    'modeled_loss_ratio_2018_pct', 'fema_funding_earthquake', 'n_faults_intersecting', 
    'centroid_to_fault_km', 'fault_length_km', 'water_area_m2', 'eq_count', 
    'eq_mean_mag', 'eq_max_mag', 'pre_1980_pct', 'built_2000_up_pct'
]

# Feature sets
feature_sets = {
    "20_features": selected_features_20,
    "10_features": selected_features_10,
    "corr_features": selected_features_corr
}


---
### ➤ Outlier Removal: Prepare Clean Dataset for Modeling


In [9]:
# Manually define outlier counties
outlier_counties = ["Mono", "San Luis Obispo", "San Diego", "Inyo", "Ventura", "Alpine", "Imperial", "San Bernardino", "Humboldt"]

# Create df without these counties
df_model = df[~df["county"].isin(outlier_counties)].copy()

print("Removed counties:", outlier_counties)
print("Original shape:", df.shape)
print("After outlier removal:", df_model.shape)


Removed counties: ['Mono', 'San Luis Obispo', 'San Diego', 'Inyo', 'Ventura', 'Alpine', 'Imperial', 'San Bernardino', 'Humboldt']
Original shape: (58, 72)
After outlier removal: (49, 72)


---
## Model Training


In [11]:
def prepare_data_for_model(feature_list, df, target_col, test_size=0.2, random_state=42):
    """
    Filters features, splits the data, and returns full and split sets.
    """
    existing_features = [col for col in feature_list if col in df.columns]
    missing_features = [col for col in feature_list if col not in df.columns]
    
    #print("Features used:", existing_features)
    #if missing_features:
     #   print("Missing features (skipped):", missing_features)

    X = df[existing_features].copy()
    y = df[target_col].astype(float).copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
    
    return X, y, X_train, X_test, y_train, y_test, existing_features


### ➤ Linear Regression


In [13]:
for name, feature_list in feature_sets.items():
    print(f"Running Linear Regression with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
)
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    lr_model = LinearRegression()
    
    lr_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", lr_model)
    ])

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=lr_pipeline,
        param_grid={},  
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=0
    )

    grid_search.fit(X, y)

    print("Best R² score (Linear Regression):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running Linear Regression with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Best R² score (Linear Regression): -0.402
-------------------------------------
Running Linear Regression with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Best R² score (Linear Regression): 0.171
-------------------------------------
Running Linear Regression with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Best R² score (Linear Regression): -8.061
-------------------------------------


---
### ➤ Ridge Regression


In [15]:
for name, feature_list in feature_sets.items():
    print(f"Running Ridge Regression with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
    )
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    ridge_model = Ridge(random_state=42)

    ridge_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", ridge_model)
    ])

    param_grid = {
        "model__alpha": [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
    }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=ridge_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score (Ridge):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running Ridge Regression with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Fitting 10 folds for each of 7 candidates, totalling 70 fits
Best parameters found:
{'model__alpha': 1.0}
Best R² score (Ridge): -0.023
-------------------------------------
Running Ridge Regression with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Fitting 10 folds for each of 7 candidates, totalling 70 fits
Best parameters found:
{'model__alpha': 50.0}
Best R² score (Ridge): 0.436
-------------------------------------
Running Ridge Regression with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Fitting 10 folds for each of 7 candidates, totalling 70 fits
Best parameters found:
{'model__alpha': 100.0}
Best R² score (Ridge): -0.456
-------------------------------------


---
### ➤ ElasticNet Regression


In [17]:
for name, feature_list in feature_sets.items():
    print(f"Running ElasticNet Regression with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
    )
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    enet_model = ElasticNet(random_state=42, max_iter=5000)

    enet_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", enet_model)
    ])

    param_grid = {
        "model__alpha": [0.01, 0.1, 1.0, 5.0, 10.0, 50.0],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
    }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=enet_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score (ElasticNet):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running ElasticNet Regression with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Fitting 10 folds for each of 30 candidates, totalling 300 fits
Best parameters found:
{'model__alpha': 1.0, 'model__l1_ratio': 0.7}
Best R² score (ElasticNet): 0.377
-------------------------------------
Running ElasticNet Regression with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Fitting 10 folds for each of 30 candidates, totalling 300 fits
Best parameters found:
{'model__alpha': 1.0, 'model__l1_ratio': 0.1}
Best R² score (ElasticNet): 0.433
-------------------------------------
Running ElasticNet Regression with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Fitting 10 folds for each of 30 candidates, totalling 300 fits
Best parameters found:
{'model__alpha': 1.0, 'model__l1_ratio': 0.5}
Best R² score (ElasticNet): 0.423
-------------------------------------


---
### ➤ Support Vector Regression (SVR)


In [19]:
for name, feature_list in feature_sets.items():
    print(f"Running Support Vector Regression (SVR) with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
    )
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    svr_model = SVR(kernel="rbf")

    svr_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", svr_model)
    ])

    param_grid = {
        "model__C": [0.1, 1, 10, 100],
        "model__epsilon": [0.01, 0.1, 0.5, 1.0]
    }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=svr_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score (SVR):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running Support Vector Regression (SVR) with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Fitting 10 folds for each of 16 candidates, totalling 160 fits
Best parameters found:
{'model__C': 10, 'model__epsilon': 0.01}
Best R² score (SVR): 0.417
-------------------------------------
Running Support Vector Regression (SVR) with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Fitting 10 folds for each of 16 candidates, totalling 160 fits
Best parameters found:
{'model__C': 100, 'model__epsilon': 0.01}
Best R² score (SVR): 0.388
-------------------------------------
Running Support Vector Regression (SVR) with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Fitting 10 folds for each of 16 candidates, totalling 160 fits
Best parameters found:
{'model__C': 100, 'model__epsilon': 0.01}
Best R² score (SVR): 0.503
-------------------------------------


---
### ➤ KNeighbors Regressor (KNN)


In [21]:
for name, feature_list in feature_sets.items():
    print(f"Running KNeighbors Regressor with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
    )
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    knn_model = KNeighborsRegressor()

    knn_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", knn_model)
    ])

    param_grid = {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=knn_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score (KNN):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running KNeighbors Regressor with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Fitting 10 folds for each of 8 candidates, totalling 80 fits
Best parameters found:
{'model__n_neighbors': 9, 'model__weights': 'distance'}
Best R² score (KNN): 0.519
-------------------------------------
Running KNeighbors Regressor with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Fitting 10 folds for each of 8 candidates, totalling 80 fits
Best parameters found:
{'model__n_neighbors': 9, 'model__weights': 'distance'}
Best R² score (KNN): 0.507
-------------------------------------
Running KNeighbors Regressor with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Fitting 10 folds for each of 8 candidates, totalling 80 fits
Best parameters found:
{'model__n_neighbors': 7, 'model__weights': 'distance'}
Best R² score (KNN): 0.543
-------------------------------------


---
### ➤ MLP Regressor


In [23]:
for name, feature_list in feature_sets.items():
    print(f"Running MLP Regressor (Neural Net) with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
    )
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    mlp_model = MLPRegressor(max_iter=10000, random_state=42)

    mlp_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", mlp_model)
    ])

    param_grid = {
        "model__hidden_layer_sizes": [(50,), (100,), (50, 50)],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.001, 0.01]
    }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=mlp_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score (MLP):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running MLP Regressor (Neural Net) with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Fitting 10 folds for each of 18 candidates, totalling 180 fits
Best parameters found:
{'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 50), 'model__learning_rate_init': 0.001}
Best R² score (MLP): -6.651
-------------------------------------
Running MLP Regressor (Neural Net) with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Fitting 10 folds for each of 18 candidates, totalling 180 fits
Best parameters found:
{'model__alpha': 0.01, 'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001}
Best R² score (MLP): 0.265
-------------------------------------
Running MLP Regressor (Neural Net) with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Fitting 10 folds for each of 18 candidates, totalling 180 fits
Best parameters found:
{'model__alpha': 0.01, 'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001}
Best R² score (MLP): -16.

---
### ➤ Random Forest Regressor


In [25]:
for name, feature_list in feature_sets.items():
    print(f"Running Random Forest Regressor with: {name}")
    
    X, y, X_train, X_test, y_train, y_test, existing_features = prepare_data_for_model(
    feature_list, df_model, TARGET
    )
    
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()) 
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, existing_features)
    ])
    
    rf_model = RandomForestRegressor(random_state=42)

    rf_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ])

    param_grid = {
        "model__n_estimators": [100, 200],
        "model__max_depth": [5, 10, None],
        "model__max_features": ["sqrt", "log2"]
    }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=rf_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score (Random Forest):", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running Random Forest Regressor with: 20_features
Train shape: (39, 20) | Test shape: (10, 20)
Fitting 10 folds for each of 12 candidates, totalling 120 fits
Best parameters found:
{'model__max_depth': 5, 'model__max_features': 'sqrt', 'model__n_estimators': 100}
Best R² score (Random Forest): 0.422
-------------------------------------
Running Random Forest Regressor with: 10_features
Train shape: (39, 10) | Test shape: (10, 10)
Fitting 10 folds for each of 12 candidates, totalling 120 fits
Best parameters found:
{'model__max_depth': 5, 'model__max_features': 'sqrt', 'model__n_estimators': 100}
Best R² score (Random Forest): 0.468
-------------------------------------
Running Random Forest Regressor with: corr_features
Train shape: (39, 26) | Test shape: (10, 26)
Fitting 10 folds for each of 12 candidates, totalling 120 fits
Best parameters found:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__n_estimators': 200}
Best R² score (Random Forest): 0.495
----------------

---
### ➤ XGBoost Regressor


In [27]:
for name, feature_list in feature_sets.items():
    print(f"Running Random Forest Regressor with: {name}")
    
    existing_features = [col for col in feature_list if col in df_model.columns]
    missing_features = [col for col in feature_list if col not in df_model.columns]
    
    X = df_model[existing_features].copy()
    y = df_model[TARGET].astype(float).copy()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)
    
    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

    preprocessor = ColumnTransformer([("num", numeric_pipeline, existing_features)])
    
    xgb_model = XGBRegressor(objective="reg:squarederror", random_state=42)

    xgb_pipeline = Pipeline([("preprocessor", preprocessor), ("model", xgb_model)])

    param_grid = {
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.01, 0.1, 0.3],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0]
        }

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

    grid_search = GridSearchCV(
        estimator=xgb_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
        )

    grid_search.fit(X, y)

    print("Best parameters found:")
    print(grid_search.best_params_)
    print("Best R² score:", round(grid_search.best_score_, 3))
    print("-------------------------------------")


Running Random Forest Regressor with: 20_features
Train shape: (39, 20)
Test shape: (10, 20)
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best parameters found:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__n_estimators': 200, 'model__subsample': 1.0}
Best R² score: 0.518
-------------------------------------
Running Random Forest Regressor with: 10_features
Train shape: (39, 10)
Test shape: (10, 10)
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best parameters found:
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best R² score: 0.436
-------------------------------------
Running Random Forest Regressor with: corr_features
Train shape: (39, 26)
Test shape: (10, 26)
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best parameters found:
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.

---

### 🖋️ Author & Attribution

This notebook is part of the **Earthquake Insurance Risk Analysis in California** project.  
Created and maintained by **Anna Piterskaya**  
Data Scientist | Geospatial & Risk Modeling | [GitHub: annapiter](https://github.com/annapiter)

© 2025 Anna Piterskaya. All rights reserved.

📄 This work is licensed under the **Creative Commons Attribution-NonCommercial 4.0 International License**.  
You may share or adapt this work **with attribution** and **non-commercial use only**.  
🔗 [View License Details](https://creativecommons.org/licenses/by-nc/4.0/)
